# 📂 지하철_따릉이_접근성_분석.ipynb

지하철 이용 규모가 비슷한 역들을 대상으로 역 주변 따릉이 대여소의 물리적 접근성이 어떻게 다른지를 확인한다. 

접근성 평가는 반경 300m 이내 대여소 분포를 기준으로 진행한다.

❉ 접근성 정의: 지하철 이용자가 하차 후 공공자전거를 이용하기 위해 대여소에 물리적으로 접근할 수 있는 정도

- 반경 300m 이내 공공자전거 대여소 수(cnt_300)
- 최소 거리 / 평균 거리도 계산 했으나 분석 및 그룹화는 cnt_300 기준으로만 진행

데이터 구성
- 지하철역 위치 데이터
- 공공자전거 대여소 위치 데이터
- 월별 지하철 하차 인원 데이터

전처리
- 환승역 및 노선 중복 제거 -> 역 단위로 통합
- haversine + BallTree를 이용한 역-대여소 거리 계산
- 지하철역 기준 반경 300m 이내 대여소 수 계산

접근성 그룹화 방식
- 반경 300m 이내 대여소가 전혀 없는 경우를 별도 그룹(0)으로 분리
- 대여소가 존재하는 역은 Low / Medium / High 로 구분

지하철 이용 규모 그룹화
- 월별 지하철 하차 인원을 기준으로 각 월 내에서 역들을 3분위(alight_group)로 구분

이용 규모 x 접근성 교차 분석
- month x alight_group x cnt_300_group 교차표 생성
- 하차 인원 규모 그룹별 접근성 분포를 행 기준 비율(normalize='index')로 비교

### 파일 불러오기

In [1]:
import pandas as pd
import numpy as np

station_raw = pd. read_csv('서울시 역사마스터 정보.csv', encoding='cp949')
bike_raw = pd.read_csv('bike_station_clean.csv')

In [2]:
print("station_raw shape:", station_raw.shape)
display(station_raw.head())

print("\n[역사 데이터 컬럼]")
print(station_raw.columns)

station_raw shape: (783, 5)


,역사_ID,역사명,호선,위도,경도
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324



[역사 데이터 컬럼]
Index(['역사_ID', '역사명', '호선', '위도', '경도'], dtype='object')


In [3]:
print("\nbike_raw shape:", bike_raw.shape)
display(bike_raw.head())

print("\n[따릉이 데이터 컬럼]")
print(bike_raw.columns)


bike_raw shape: (2764, 5)


,대여소번호,대여소명,위도,경도,대여소구분
0,ST-4,102. 망원역 1번출구 앞,37.555649,126.910629,RAK_002
1,ST-5,103. 망원역 2번출구 앞,37.554951,126.910835,RAK_002
2,ST-6,104. 합정역 1번출구 앞,37.550739,126.915085,RAK_002
3,ST-7,105. 합정역 5번출구 앞,37.550007,126.914825,RAK_002
4,ST-8,106. 합정역 7번출구 앞,37.548645,126.912827,RAK_002



[따릉이 데이터 컬럼]
Index(['대여소번호', '대여소명', '위도', '경도', '대여소구분'], dtype='object')


## 분석에 필요한 컬럼만 남기고 이름 통일

In [4]:
# 역사 데이터: 필요한 데이터 + 영문 이름
station = station_raw.rename(
    columns={
        '역사명': 'station_name',
        '위도': 'lat',
        '경도': 'lon'
    }
)[['station_name', 'lat','lon']]

# 따릉이 데이터
bike = bike_raw.rename(
    columns={
        '대여소번호': 'bike_id',
        '위도': 'lat',
        '경도': 'lon'
        }
)[['bike_id', 'lat','lon']]

print(station.head())
print(bike.head())


  station_name       lat        lon
0           동탄  37.20034  127.09569
1           구성  37.29913  127.10389
2           성남  37.39467  127.12058
3           수서  37.48637  127.10161
4           삼성  37.50887  127.06324
  bike_id        lat         lon
0    ST-4  37.555649  126.910629
1    ST-5  37.554951  126.910835
2    ST-6  37.550739  126.915085
3    ST-7  37.550007  126.914825
4    ST-8  37.548645  126.912827


### 위도 경도 타입 확인 / 숫자형 변환

In [5]:
station['lat'] = pd.to_numeric(station['lat'], errors='coerce')
station['lon'] = pd.to_numeric(station['lon'], errors='coerce')

bike['lat'] = pd.to_numeric(bike['lat'], errors='coerce')
bike['lon'] = pd.to_numeric(bike['lon'], errors='coerce')

print('station dtypes:')
print(station.dtypes)

print('\nbike dtypes:')
print(bike.dtypes)

station dtypes:
station_name     object
lat             float64
lon             float64
dtype: object

bike dtypes:
bike_id     object
lat        float64
lon        float64
dtype: object


### 결측치 제거

In [6]:
station = station.dropna(subset=['station_name', 'lat', 'lon']).copy()
bike = bike.dropna(subset=['bike_id', 'lat', 'lon']).copy()

print(station.shape)
print(bike.shape)

(783, 3)
(2764, 3)


### 환승역(중복 역사명) 정리

접근성 분석은 역 단위이기 때문에 하나로 묶음

In [7]:
station_u = (station.groupby('station_name', as_index=False).agg(lat=('lat', 'mean'), lon=('lon', 'mean')))

station_u.head()

,station_name,lat,lon
0,4.19민주묘지,37.649502,127.013684
1,가능,37.748577,127.044213
2,가락시장,37.492566,127.118077
3,가산디지털단지,37.480959,126.882619
4,가양,37.561391,126.854456


### BallTree 만들기

In [8]:
from sklearn.neighbors import BallTree

earth_radius_m = 6371000 # 지구 반지름(m)

# 라디안으로 변환
bike_rad =np.radians(bike[['lat', 'lon']].to_numpy())
stn_rad =np.radians(station_u[['lat', 'lon']].to_numpy())

# 대여소 좌표로 트리 생성
tree = BallTree(bike_rad, metric='haversine')


### 반경 300m 내 대여소 수 계산

In [9]:
r300 = 300 / earth_radius_m

cnt_300 = tree.query_radius(stn_rad, r=r300, count_only=True)

station_u['cnt_300'] = cnt_300

station_u[['station_name', 'cnt_300']].head()

,station_name,cnt_300
0,4.19민주묘지,3
1,가능,0
2,가락시장,5
3,가산디지털단지,3
4,가양,3


### 300m 기준 평균•최소 거리 계산

In [10]:
inds_300, dists_300 = tree.query_radius(stn_rad, r=r300, return_distance=True, sort_results=True)

min_dist_300 = []
avg_dist_300 = []

for dists in dists_300:
    if len(dists ) == 0:
        min_dist_300.append(np.nan)
        avg_dist_300.append(np.nan)
    else:
        d_m = dists * earth_radius_m
        min_dist_300.append(d_m.min())
        avg_dist_300.append(d_m.mean())

access_300 = station_u[['station_name', 'cnt_300']].copy()
access_300['min_dist_300m'] = np.round(min_dist_300, 1)
access_300['avg_dist_300m'] = np.round(avg_dist_300, 1)

access_300.head()

,station_name,cnt_300,min_dist_300m,avg_dist_300m
0,4.19민주묘지,3,17.9,79.0
1,가능,0,NaN,NaN
2,가락시장,5,20.1,112.4
3,가산디지털단지,3,88.3,109.7
4,가양,3,50.5,128.1


In [11]:
access_300.describe()

,cnt_300,min_dist_300m,avg_dist_300m
count,655.000000,300.000000,300.000000
mean,1.461069,76.144333,143.391000
std,1.882138,53.164294,54.057294
min,0.000000,2.800000,10.400000
25%,0.000000,35.700000,107.950000
50%,0.000000,64.500000,147.150000
75%,3.000000,104.200000,182.350000
max,8.000000,294.000000,294.000000


### 접근성 그룹 만들기

지하철역 반경 300m 내 따릉이 대여소 수('cnt_300')를 기준으로 접근성 수준을 다음과 같이 구분한다.

- 반경 300m 내 대여소가 없는 경우: 0(없음)
- 반경 300m 내 대여소가 1개 이상 있는 경우: 분위수 기준으로 'Q1~Q4' 그룹으로 분류

In [12]:
# 300m 접근성 그룹(0 / L / M / H)
access_300['cnt_300_group']= '0(없음)'

mask = access_300['cnt_300'] > 0
access_300.loc[mask, 'cnt_300_group'] = pd.qcut(
    access_300.loc[mask, 'cnt_300'],
    q=3,
    labels=['L', 'M', 'H'],
    duplicates='drop')

access_300[['station_name', 'cnt_300', 'cnt_300_group']].head(10)

,station_name,cnt_300,cnt_300_group
0,4.19민주묘지,3,M
1,가능,0,0(없음)
2,가락시장,5,H
3,가산디지털단지,3,M
4,가양,3,M
5,가오리,3,M
6,가재울,0,0(없음)
7,가정(루원시티),0,0(없음)
8,가정중앙시장,0,0(없음)
9,가좌,4,M


In [13]:
# 분포 확인
access_300['cnt_300_group'].value_counts().sort_index()

cnt_300_group
0(없음)    355
H         58
L        111
M        131
Name: count, dtype: int64

### 월별 하차 인원(지하철 이용 규모)

In [14]:
import pandas as pd

# 월별 시트 전부 읽어서 합치기
path = 'subway_2025.xlsx'
xls = pd.ExcelFile(path)

dfs = []
for sh in xls.sheet_names:
    tmp = pd.read_excel(xls, sheet_name=sh)
    tmp['month'] = sh
    dfs.append(tmp)

subway_raw = pd.concat(dfs, ignore_index=True)
subway_raw.head()

,사용일자,노선명,역명,승차총승객수,하차총승객수,등록일자,month
0,20250101,수인선,송도,1453,1321,20250104,2501
1,20250101,4호선,창동,12477,13408,20250104,2501
2,20250101,4호선,쌍문,12792,12199,20250104,2501
3,20250101,4호선,수유(강북구청),17606,17442,20250104,2501
4,20250101,4호선,미아(서울사이버대학),6819,6532,20250104,2501


In [15]:
# 필요한 컬럼만 정리
subway = subway_raw.rename(columns={
    '역명' : 'station_name',
    '하차총승객수': 'alight'
})[['month', 'station_name', 'alight']].copy()

# 타입 정리
subway['station_name'] = subway['station_name'].astype(str).str.strip()
subway['alight'] = pd.to_numeric(subway['alight'], errors='coerce')

subway.head()

,month,station_name,alight
0,2501,송도,1321
1,2501,창동,13408
2,2501,쌍문,12199
3,2501,수유(강북구청),17442
4,2501,미아(서울사이버대학),6532


### 월・역별 하차 인원 합산

In [16]:
subway_m = (subway.dropna(subset=['station_name', 'alight'])
            .groupby(['month', 'station_name'], as_index=False)['alight'].sum())

subway_m.head()

,month,station_name,alight
0,2501,4.19민주묘지,74425
1,2501,가능,160932
2,2501,가락시장,500540
3,2501,가산디지털단지,1521317
4,2501,가양,574129


### 월별 하차 인원 그룹 만들기
- 월마다 따로 분위수 나눔

In [17]:
subway_m['alight_group'] = subway_m.groupby('month')['alight'].transform(
    lambda s: pd.qcut(s, q=3, labels=['L1(적음)', 'L2', 'L3(많음)'], duplicates='drop')
)
subway_m.head()

,month,station_name,alight,alight_group
0,2501,4.19민주묘지,74425,L1(적음)
1,2501,가능,160932,L2
2,2501,가락시장,500540,L3(많음)
3,2501,가산디지털단지,1521317,L3(많음)
4,2501,가양,574129,L3(많음)


### 접근성(300m) 데이터와 결합

In [18]:
subway_acc_300 = subway_m.merge(
    access_300[['station_name', 'cnt_300', 'cnt_300_group']], 
    on='station_name', how='left', indicator=True)

subway_acc_300['_merge'].value_counts()

_merge
both          6224
left_only      121
right_only       0
Name: count, dtype: int64

### 같은 이용 규모 안에서 접근성 비교

In [19]:
# 월별 x 이용 규묘 x 접근성 분포
pd.crosstab([subway_acc_300['month'], subway_acc_300['alight_group']],
            subway_acc_300['cnt_300_group'], normalize='index')

cnt_300_group          0(없음)         H         L         M
month alight_group                                        
2501  L1(적음)        0.699422  0.028902  0.150289  0.121387
      L2            0.404624  0.080925  0.289017  0.225434
      L3(많음)        0.215116  0.215116  0.203488  0.366279
2502  L1(적음)        0.699422  0.028902  0.150289  0.121387
      L2            0.410405  0.069364  0.289017  0.231214
      L3(많음)        0.209302  0.226744  0.203488  0.360465
2503  L1(적음)        0.699422  0.028902  0.156069  0.115607
      L2            0.404624  0.080925  0.277457  0.236994
      L3(많음)        0.215116  0.215116  0.209302  0.360465
2504  L1(적음)        0.693642  0.034682  0.156069  0.115607
      L2            0.421965  0.069364  0.277457  0.231214
      L3(많음)        0.202312  0.219653  0.208092  0.369942
2505  L1(적음)        0.693642  0.028902  0.156069  0.121387
      L2            0.419540  0.068966  0.281609  0.229885
      L3(많음)        0.209302  0.220930  0.203488  0.366279
2506  L1(적음)        0.699422  0.028902  0.156069  0.115607
      L2            0.408046  0.068966  0.281609  0.241379
      L3(많음)        0.215116  0.220930  0.203488  0.360465
2507  L1(적음)        0.712644  0.022989  0.149425  0.114943
      L2            0.396552  0.074713  0.287356  0.241379
      L3(많음)        0.215116  0.220930  0.203488  0.360465
2508  L1(적음)        0.699422  0.028902  0.150289  0.121387
      L2            0.408046  0.068966  0.287356  0.235632
      L3(많음)        0.215116  0.220930  0.203488  0.360465
2509  L1(적음)        0.705202  0.028902  0.144509  0.121387
      L2            0.408046  0.063218  0.293103  0.235632
      L3(많음)        0.204678  0.228070  0.204678  0.362573
2510  L1(적음)        0.689655  0.028736  0.160920  0.120690
      L2            0.421965  0.063584  0.277457  0.236994
      L3(많음)        0.209302  0.226744  0.203488  0.360465
2511  L1(적음)        0.706897  0.028736  0.143678  0.120690
      L2            0.408046  0.068966  0.293103  0.229885
      L3(많음)        0.204678  0.222222  0.204678  0.368421
2512  L1(적음)        0.699422  0.028902  0.150289  0.121387
      L2            0.408046  0.068966  0.287356  0.235632
      L3(많음)        0.210526  0.222222  0.204678  0.362573

### 월별 표를 '전체 요약'으로 압축하기

In [20]:
# 월을 빼고 이용규모별 접근성 분포(전체)
overall = pd.crosstab(
    subway_acc_300['alight_group'],
    subway_acc_300['cnt_300_group'],
    normalize='index'
)

# %로
(overall * 100).round(1)

cnt_300_group,0(없음),H,L,M
alight_group,,,,
L1(적음),70.0,2.9,15.2,11.9
L2,41.0,7.1,28.5,23.4
L3(많음),21.0,22.2,20.5,36.3


### 핵심 비율만 추출

In [21]:
summary = (overall * 100).round(1)

summary.loc[:, ['0(없음)', 'H']]

cnt_300_group,0(없음),H
alight_group,,
L1(적음),70.0,2.9
L2,41.0,7.1
L3(많음),21.0,22.2


In [22]:
overall = pd.crosstab(
    subway_acc_300['alight_group'],
    subway_acc_300['cnt_300_group'],
    normalize='index'
)

In [23]:
tableau_df = (
    pd.crosstab(
        [subway_acc_300['month'], subway_acc_300['alight_group']],
        subway_acc_300['cnt_300_group'],
        normalize='index'
    )
    .reset_index()
    .melt(
        id_vars=['month', 'alight_group'],
        var_name='cnt_300_group',
        value_name='ratio'
    )
)

tableau_df['ratio'] = (tableau_df['ratio'] * 100).round(1)


In [24]:
tableau_df.head()

,month,alight_group,cnt_300_group,ratio
0,2501,L1(적음),0(없음),69.9
1,2501,L2,0(없음),40.5
2,2501,L3(많음),0(없음),21.5
3,2502,L1(적음),0(없음),69.9
4,2502,L2,0(없음),41.0


In [25]:
tableau_df.to_csv(
    'subway_bike_access_300m_tableau.csv',
    index=False,
    encoding='utf-8-sig'
)


### 카이제곱 검정
- subway_acc_300

    -> 역 x 월 단위 데이터

    -> alight_group: 하차 규모 그룹(L1/L2/L3)

    -> cnt_300: 300m 내 대여소 개수(숫자)

    -> cnt_300_group: 접근성 그룹(0/L/M/H)

- 월별 변화 분석이 아닌, 전체 기간 기준 하차 인원 규모 그룹과 접근성 분포의 관계를 확인하기 위해 수행

In [26]:
# 결측치 및 0 처리
import numpy as np

# 숫자형: 값 없으면  0
subway_acc_300['cnt_300'] = subway_acc_300['cnt_300'].fillna(0).astype(int)

# 접근성 그룹: cnt_300 == 0 인 경우는 모두 '0(없음)'으로 통일
subway_acc_300['cnt_300_group'] = np.where(subway_acc_300['cnt_300'] == 0, '0(없음)', subway_acc_300['cnt_300_group'])

In [27]:
# 카이제곱 검정용 데이터 추출
raw = subway_acc_300[['station_name', 'alight_group', 'cnt_300_group']].dropna(subset=['alight_group', 'cnt_300_group']).copy()

# 중복행 있을 경우 제거
raw = raw.drop_duplicates(subset=['station_name'])

In [28]:
# 교차표 생성
# 행: 하차 규모 그룹 | 열: 접근성 그룹 | 값: 관측치 수
import pandas as pd

table = pd.crosstab(raw['alight_group'], raw['cnt_300_group'])
table

cnt_300_group,0(없음),H,L,M
alight_group,,,,
L1(적음),126,5,26,21
L2,74,14,50,40
L3(많음),41,37,35,63


In [29]:
# 카이제곱 독립성 검정
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(table)
chi2, p

(np.float64(104.3917364876092), np.float64(3.0373979530215613e-20))

In [30]:
# 카이제곱 검정 가정 확인 - 모든 기대빈도가 5이상인지
expected.min()

np.float64(18.526315789473685)

하차 규모가 작을수록 -> 대여소가 아예 없는 역 비중이 높음
하차 규모가 클수록 -> M・H 비중 증가

#### Cramér's V (효과크기)
- 카이제곱 검정 결과를 바탕으로 두 범주형 변수 간 관계의 영향력을 0~1 사이의 값으로 나타내는 지표

- 표본 수가 많아 p-value가 매우 작게 나타날 수 있다고 판단하여, 하차 인원 규모 그룹과 300m 이내 공공자전거 대여소 수 그룹 간 관계의 통계적 유의성과 함께 관계의 강도를 확인하고자 추가로 계산했다.

❉ 일반적인 해석 기준
- 0.1 미만: 약한 관계
- 0.1 ~ 0.3: 약 ~ 중간 관계
- 0.3 ~ 0.5: 중간 관계
- 0.5 이상: 강한 관계

In [31]:
n = table.to_numpy().sum()
r, k = table.shape

cramers_v = np.sqrt(chi2 / (n * (min(r-1, k-1))))

cramers_v

np.float64(0.31322920408682486)

In [32]:
# 비율표
table_ratio = table.div(table.sum(axis=1), axis=0)

(table_ratio * 100).round(1)

cnt_300_group,0(없음),H,L,M
alight_group,,,,
L1(적음),70.8,2.8,14.6,11.8
L2,41.6,7.9,28.1,22.5
L3(많음),23.3,21.0,19.9,35.8


In [33]:
# 표준화
residuals = (table - expected) / np.sqrt(expected)

residuals = pd.DataFrame(residuals, index=table.index, columns=table.columns).round(2)

residuals

cnt_300_group,0(없음),H,L,M
alight_group,,,,
L1(적음),5.05,-3.17,-1.83,-3.18
L2,-0.74,-1.09,2.11,-0.23
L3(많음),-4.34,4.29,-0.28,3.43


### 결론
지하철역의 하차 인원 규모 그룹(alight_group)과 300m 이내 공공자전거 대여소 수 그룹(cnt_300_group) 간의 관계를 확인하기 위해 교차표 분석과 카이제곱 검정을 수행했다.
교차표 및 비율을 비교한 결과, 하차 인원 규모가 작은 역에서는 300m 이내에 공공자전거 대여소가 전혀 없는 경우의 비율이 가장 높게 나타났다. 반면, 하차 인원 규모가 커질수록 공공자전거 대여소 수가 중간(M) 또는 높은(H) 수준에 해당하는 역의 비율이 증가하는 경향이 나타났다. 이러한 분포 차이는 하차 인원 규모 그룹별 대여소 수 구성 비율을 비교하는 과정에서 확인할 수 있었다.

카이제곱 검정 결과, 두 변수 간 관계는 통계적으로 유의하게 나타났으며(p < 0.001), 하차 인원 규모 그룹과 공공자전거 대여소 수 그룹이 서로 독립적이라는 가설은 기각되었다. 그러나 p-value가 매우 작게 나타나 단순한 유의성 여부만으로는 충분하지 않다고 판단하여, 두 변수 간 연관성의 크기를 확인하기 위해 Cramér's V를 추가로 계산했다. 그 결과 Cramér's V 값은 0.31로 나타났으며, 이는 두 범주형 변수 간에 약하지 않은 수준의 연관성이 존재함을 의미한다.


# 📂 따릉이이용수준분석_거치대제외.ipynb

# 데이터 전처리

## 지하철별 하차인원 확인

In [34]:
# 파일 불러오기
import pandas as pd

# 각 시트(월별로 나눠져있음)에 월별코드 추가
station_df_01 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 0)
station_df_01['월별코드'] = 2501

station_df_02 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 1)
station_df_02['월별코드'] = 2502

station_df_03 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 2)
station_df_03['월별코드'] = 2503

station_df_04 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 3)
station_df_04['월별코드'] = 2504

station_df_05 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 4)
station_df_05['월별코드'] = 2505

station_df_06 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 5)
station_df_06['월별코드'] = 2506

station_df_07 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 6)
station_df_07['월별코드'] = 2507

station_df_08 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 7)
station_df_08['월별코드'] = 2508

station_df_09 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 8)
station_df_09['월별코드'] = 2509

station_df_10 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 9)
station_df_10['월별코드'] = 2510

station_df_11 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 10)
station_df_11['월별코드'] = 2511

station_df_12 = pd.read_excel('서울시_지하철호선별_역별_승하차_인원.xlsx',sheet_name = 11)
station_df_12['월별코드'] = 2512

station_all = pd.concat([
    station_df_01,
    station_df_02,
    station_df_03,
    station_df_04,
    station_df_05,
    station_df_06,
    station_df_07,
    station_df_08,
    station_df_09,
    station_df_10,
    station_df_11,
    station_df_12
], ignore_index=True)

station_all

,사용일자,노선명,역명,승차총승객수,하차총승객수,등록일자,월별코드
0,20250101,수인선,송도,1453,1321,20250104,2501
1,20250101,4호선,창동,12477,13408,20250104,2501
2,20250101,4호선,쌍문,12792,12199,20250104,2501
3,20250101,4호선,수유(강북구청),17606,17442,20250104,2501
4,20250101,4호선,미아(서울사이버대학),6819,6532,20250104,2501
...,...,...,...,...,...,...,...
224611,20251231,3호선,양재(서초구청),37579,37715,20260103,2512
224612,20251231,3호선,매봉,13546,12833,20260103,2512
224613,20251231,3호선,도곡,6117,6009,20260103,2512
224614,20251231,3호선,대치,12096,12175,20260103,2512


In [35]:
# 월별코드, 역명이 똑같은 행의 하차총승객수 합
station_all_df = station_all.groupby(['월별코드','역명'], as_index = False)['하차총승객수'].sum()

In [36]:
# 역명 중복 행 확인
duplicates = station_all_df[station_all_df.duplicated(subset=['역명'],keep=False)]
duplicates_sorted = duplicates.sort_values('역명')
duplicates_sorted.head(13)

,월별코드,역명,하차총승객수
0,2501,4.19민주묘지,74425
5817,2512,4.19민주묘지,101491
5288,2511,4.19민주묘지,114850
4759,2510,4.19민주묘지,106324
4231,2509,4.19민주묘지,119453
3702,2508,4.19민주묘지,81455
2643,2506,4.19민주묘지,103436
2114,2505,4.19민주묘지,118469
1584,2504,4.19민주묘지,124599
1056,2503,4.19민주묘지,120713


월별 역별로 필요한 행과 열만 남은 걸 확인할 수 있다

In [37]:
# 역 별로 위도, 경도 추가 하기 

# 위도, 경도가 있는 데이터 가져오기 -> station_all_df에 추가
station_location = pd.read_csv('서울시 역사마스터 정보.csv',encoding='cp949')
station_location.head()

,역사_ID,역사명,호선,위도,경도
0,9010,동탄,수도권 광역급행철도,37.20034,127.09569
1,9009,구성,수도권 광역급행철도,37.29913,127.10389
2,9008,성남,수도권 광역급행철도,37.39467,127.12058
3,9007,수서,수도권 광역급행철도,37.48637,127.10161
4,9006,삼성,수도권 광역급행철도,37.50887,127.06324


In [38]:
# station_all_df의 역명과 station_location의 역사명이 똑같은 곳에 위도, 경도 추가

merged_df = station_all_df.merge(
    station_location[["역사명","위도","경도"]],
    left_on="역명",
    right_on="역사명",
    how="left"
)


In [39]:
merged_df[merged_df['위도'].isna()]

,월별코드,역명,하차총승객수,역사명,위도,경도
103,2501,낙성대(강감찬),697298,NaN,NaN,NaN
129,2501,당고개,224671,NaN,NaN,NaN
167,2501,동대문역사문화공원(DDP),978381,NaN,NaN,NaN
186,2501,마곡나루(서울식물원),616616,NaN,NaN,NaN
282,2501,상봉,615241,NaN,NaN,NaN
...,...,...,...,...,...,...
7531,2512,용마산(용마폭포공원),173720,NaN,NaN,NaN
7578,2512,자양(뚝섬한강공원),252844,NaN,NaN,NaN
7662,2512,평택지제,130088,NaN,NaN,NaN
7666,2512,하남시청(덕풍·신장),261966,NaN,NaN,NaN


In [40]:
station_df = merged_df.dropna()
station_df

,월별코드,역명,하차총승객수,역사명,위도,경도
0,2501,4.19민주묘지,74425,4.19민주묘지,37.649502,127.013684
1,2501,가능,160932,가능,37.748577,127.044213
2,2501,가락시장,500540,가락시장,37.492888,127.118398
3,2501,가락시장,500540,가락시장,37.492245,127.117757
4,2501,가산디지털단지,1521317,가산디지털단지,37.480338,126.882656
...,...,...,...,...,...,...
7695,2512,회룡,402193,회룡,37.724416,127.047360
7696,2512,회현(남대문시장),965626,회현(남대문시장),37.558514,126.978246
7697,2512,효창공원앞,327673,효창공원앞,37.539233,126.961384
7698,2512,효창공원앞,327673,효창공원앞,37.538579,126.962210


In [41]:
# 중복행 확인
station_df[
    station_df.duplicated(subset=["역명","월별코드"], keep=False)
]

,월별코드,역명,하차총승객수,역사명,위도,경도
2,2501,가락시장,500540,가락시장,37.492888,127.118398
3,2501,가락시장,500540,가락시장,37.492245,127.117757
4,2501,가산디지털단지,1521317,가산디지털단지,37.480338,126.882656
5,2501,가산디지털단지,1521317,가산디지털단지,37.481581,126.882581
13,2501,강남,2110793,강남,37.496837,127.028104
...,...,...,...,...,...,...
7686,2512,홍대입구,3215809,홍대입구,37.556790,126.923708
7694,2512,회룡,402193,회룡,37.725006,127.047073
7695,2512,회룡,402193,회룡,37.724416,127.047360
7697,2512,효창공원앞,327673,효창공원앞,37.539233,126.961384


In [42]:
# 위도 경도 평균으로 계산
station_u = (station_df.groupby('역명', as_index=False).agg(위도=('위도', 'mean'), 경도=('경도', 'mean')))

# station_df에 위도 경도 업데이트
station_df = station_df.drop(columns=["위도","경도"])
station_df = station_df.merge(
    station_u,
    on="역명",
    how="left"
)
station_df = station_df.drop_duplicates(
    subset=["역명","월별코드"]
)
station_df

,월별코드,역명,하차총승객수,역사명,위도,경도
0,2501,4.19민주묘지,74425,4.19민주묘지,37.649502,127.013684
1,2501,가능,160932,가능,37.748577,127.044213
2,2501,가락시장,500540,가락시장,37.492567,127.118077
4,2501,가산디지털단지,1521317,가산디지털단지,37.480960,126.882618
6,2501,가양,574129,가양,37.561391,126.854456
...,...,...,...,...,...,...
7572,2512,회기,803395,회기,37.589460,127.057583
7573,2512,회룡,402193,회룡,37.724711,127.047216
7575,2512,회현(남대문시장),965626,회현(남대문시장),37.558514,126.978246
7576,2512,효창공원앞,327673,효창공원앞,37.538906,126.961797


## 따릉이 대여소 이용건수 정리

In [43]:
bike_df_01 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=0)

# 대여소 번호가 같으면 이용건수 합치기
bike_sum_01 = bike_df_01.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_01['월별코드'] = 2501

#12월까지 반복 후 데이터 합치기
bike_df_02 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=1)
bike_sum_02 = bike_df_02.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_02['월별코드'] = 2502

bike_df_03 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=2)
bike_sum_03 = bike_df_03.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_03['월별코드'] = 2503

bike_df_04 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=3)
bike_sum_04 = bike_df_04.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_04['월별코드'] = 2504

bike_df_05 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=4)
bike_sum_05 = bike_df_05.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_05['월별코드'] = 2505

bike_df_06 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=5)
bike_sum_06 = bike_df_06.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_06['월별코드'] = 2506

bike_df_07 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=6)
bike_sum_07 = bike_df_07.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_07['월별코드'] = 2507

bike_df_08 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=7)
bike_sum_08 = bike_df_08.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_08['월별코드'] = 2508

bike_df_09 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=8)
bike_sum_09 = bike_df_09.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_09['월별코드'] = 2509

bike_df_10 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=9)
bike_sum_10 = bike_df_10.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_10['월별코드'] = 2510

bike_df_11 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=10)
bike_sum_11 = bike_df_11.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_11['월별코드'] = 2511

bike_df_12 = pd.read_excel('따릉이이용로그월별시트.xlsx',sheet_name=11)
bike_sum_12 = bike_df_12.groupby(['대여소번호'],as_index = False)['이용건수'].sum()
bike_sum_12['월별코드'] = 2512

bike_all = pd.concat([
    bike_sum_01,
    bike_sum_02,
    bike_sum_03,
    bike_sum_04,
    bike_sum_05,
    bike_sum_06,
    bike_sum_07,
    bike_sum_08,
    bike_sum_09,
    bike_sum_10,
    bike_sum_11,
    bike_sum_12
], ignore_index=True)

bike_all

,대여소번호,이용건수,월별코드
0,102,1563,2501
1,103,1091,2501
2,104,788,2501
3,105,459,2501
4,106,666,2501
...,...,...,...
32962,6185,59,2512
32963,6187,352,2512
32964,6188,546,2512
32965,6189,362,2512


In [44]:
# 자전거 위치 정보(위도, 경도) 추가  (불필요 컬럼은 사용하지 않음)
bike_location = pd.read_excel('공공자전거 대여소 정보.xlsx', sheet_name=3)

# 필요한 컬럼만 남기기 (대여소번호/위도/경도)
keep_cols = [c for c in ['대여소번호','위도','경도'] if c in bike_location.columns]
bike_location = bike_location[keep_cols].copy()

# 타입 정리
bike_location['대여소번호'] = pd.to_numeric(bike_location['대여소번호'], errors='coerce')
bike_location['위도'] = pd.to_numeric(bike_location['위도'], errors='coerce')
bike_location['경도'] = pd.to_numeric(bike_location['경도'], errors='coerce')

bike_location = bike_location.dropna(subset=['대여소번호','위도','경도'])
bike_location

,대여소번호,위도,경도
0,102,37.555649,126.910629
1,103,37.554951,126.910835
2,104,37.550629,126.914986
3,105,37.550007,126.914825
4,106,37.548645,126.912827
...,...,...,...
2794,6185,37.573410,126.843452
2795,6187,37.555347,126.820724
2796,6188,37.556190,126.864639
2797,6189,37.564484,126.848305


In [45]:
bike_merged_df = bike_all.merge(
    bike_location[['대여소번호','위도','경도']],
    on='대여소번호',
    how='left'
)
bike_merged_df

,대여소번호,이용건수,월별코드,위도,경도
0,102,1563,2501,37.555649,126.910629
1,103,1091,2501,37.554951,126.910835
2,104,788,2501,37.550629,126.914986
3,105,459,2501,37.550007,126.914825
4,106,666,2501,37.548645,126.912827
...,...,...,...,...,...
32962,6185,59,2512,37.573410,126.843452
32963,6187,352,2512,37.555347,126.820724
32964,6188,546,2512,37.556190,126.864639
32965,6189,362,2512,37.564484,126.848305


In [46]:
# 결측치 확인
bike_merged_df[bike_merged_df['위도'].isna()]

,대여소번호,이용건수,월별코드,위도,경도
345,538,217,2501,NaN,NaN
431,659,1356,2501,NaN,NaN
432,660,1178,2501,NaN,NaN
627,961,511,2501,NaN,NaN
697,1054,154,2501,NaN,NaN
...,...,...,...,...,...
29824,4605,301,2511,NaN,NaN
29989,4911,344,2511,NaN,NaN
31211,1513,83,2512,NaN,NaN
31576,2177,2,2512,NaN,NaN


결측치 제거

In [47]:
bike_df = bike_merged_df.dropna()
bike_df

,대여소번호,이용건수,월별코드,위도,경도
0,102,1563,2501,37.555649,126.910629
1,103,1091,2501,37.554951,126.910835
2,104,788,2501,37.550629,126.914986
3,105,459,2501,37.550007,126.914825
4,106,666,2501,37.548645,126.912827
...,...,...,...,...,...
32962,6185,59,2512,37.573410,126.843452
32963,6187,352,2512,37.555347,126.820724
32964,6188,546,2512,37.556190,126.864639
32965,6189,362,2512,37.564484,126.848305


In [48]:
# 중복값 확인 
bike_df[bike_df.duplicated()]

,대여소번호,이용건수,월별코드,위도,경도


## 지하철 역 반경 300m 이내의 있는 따릉이 대여소의 이용건수의 합

In [49]:
import numpy as np
# 거리 함수 정의
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meter
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# 결과 컬럼 생성 (300m 내 따릉이 '이용건수'만 계산)
station_df['300m_따릉이이용건수'] = 0

# 월별코드 기준으로 반복
for month in station_df['월별코드'].unique():
    st = station_df[station_df['월별코드'] == month]
    bk = bike_df[bike_df['월별코드'] == month]

    for idx, row in st.iterrows():
        dist = haversine(
            row['위도'], row['경도'],
            bk['위도'].values, bk['경도'].values
        )
        total_usage = bk.loc[dist <= 300, '이용건수'].sum()
        station_df.loc[idx, '300m_따릉이이용건수'] = total_usage


In [50]:
station_df

,월별코드,역명,하차총승객수,역사명,위도,경도,300m_따릉이이용건수
0,2501,4.19민주묘지,74425,4.19민주묘지,37.649502,127.013684,767
1,2501,가능,160932,가능,37.748577,127.044213,0
2,2501,가락시장,500540,가락시장,37.492567,127.118077,2934
4,2501,가산디지털단지,1521317,가산디지털단지,37.480960,126.882618,4714
6,2501,가양,574129,가양,37.561391,126.854456,7011
...,...,...,...,...,...,...,...
7572,2512,회기,803395,회기,37.589460,127.057583,0
7573,2512,회룡,402193,회룡,37.724711,127.047216,0
7575,2512,회현(남대문시장),965626,회현(남대문시장),37.558514,126.978246,675
7576,2512,효창공원앞,327673,효창공원앞,37.538906,126.961797,1112


In [51]:
# 월별 역별로 이용건수 합계 확인
duplicates = station_df[station_df.duplicated(subset=['역명'],keep=False)]
duplicates_sorted = duplicates.sort_values('역명')
duplicates_sorted.head()

,월별코드,역명,하차총승객수,역사명,위도,경도,300m_따릉이이용건수
0,2501,4.19민주묘지,74425,4.19민주묘지,37.649502,127.013684,767
6949,2512,4.19민주묘지,101491,4.19민주묘지,37.649502,127.013684,863
6317,2511,4.19민주묘지,114850,4.19민주묘지,37.649502,127.013684,1702
5685,2510,4.19민주묘지,106324,4.19민주묘지,37.649502,127.013684,1767
5055,2509,4.19민주묘지,119453,4.19민주묘지,37.649502,127.013684,2013


In [52]:
# 분석에 필요한 월별코드, 역명, 하차총승객수, 이용건수 컬럼 제외 제거하기
station_df = station_df.drop(['역사명', '위도','경도'], axis = 1)
station_df

,월별코드,역명,하차총승객수,300m_따릉이이용건수
0,2501,4.19민주묘지,74425,767
1,2501,가능,160932,0
2,2501,가락시장,500540,2934
4,2501,가산디지털단지,1521317,4714
6,2501,가양,574129,7011
...,...,...,...,...
7572,2512,회기,803395,0
7573,2512,회룡,402193,0
7575,2512,회현(남대문시장),965626,675
7576,2512,효창공원앞,327673,1112


## 지하철 이용 규모 대비 따릉이 이용건수

In [53]:
station_df['하차인원대비따릉이이용비율[%]'] = (station_df['300m_따릉이이용건수'] / station_df['하차총승객수']) * 100
station_df.head()

,월별코드,역명,하차총승객수,300m_따릉이이용건수,하차인원대비따릉이이용비율[%]
0,2501,4.19민주묘지,74425,767,1.030568
1,2501,가능,160932,0,0.000000
2,2501,가락시장,500540,2934,0.586167
4,2501,가산디지털단지,1521317,4714,0.309863
6,2501,가양,574129,7011,1.221154


In [54]:
# 따릉이 이용 비율을 활용해 그룹화 / 이용비율을 3분위로 나눔 (0(300m 내에 300m 내 이용건수가 0인 경우), low, middle, high)

zero_mask = station_df['300m_따릉이이용건수'] == 0
non_zero_mask = station_df.loc[~zero_mask].copy()
non_zero_mask['이용비율_그룹'] = pd.qcut(
    non_zero_mask['하차인원대비따릉이이용비율[%]'],
    q = 3,
    labels = ['low','middle', 'high']
)
station_df['이용비율_그룹'] = '0'
station_df.loc[~zero_mask, '이용비율_그룹'] = non_zero_mask['이용비율_그룹']

In [55]:
station_df

,월별코드,역명,하차총승객수,300m_따릉이이용건수,하차인원대비따릉이이용비율[%],이용비율_그룹
0,2501,4.19민주묘지,74425,767,1.030568,middle
1,2501,가능,160932,0,0.000000,0
2,2501,가락시장,500540,2934,0.586167,middle
4,2501,가산디지털단지,1521317,4714,0.309863,low
6,2501,가양,574129,7011,1.221154,high
...,...,...,...,...,...,...
7572,2512,회기,803395,0,0.000000,0
7573,2512,회룡,402193,0,0.000000,0
7575,2512,회현(남대문시장),965626,675,0.069903,low
7576,2512,효창공원앞,327673,1112,0.339363,low


## 월별로 지하철 이용 규모 그룹화

In [56]:
# 월별로 자하철 이용 규모를 활용해 대형, 중형, 소형 그룹으로 그룹화
station_df['이용규모_그룹'] = (
    station_df.groupby('월별코드')['하차총승객수']
    .transform(lambda x: pd.qcut(
        x,
        q = 3,
        labels=['소형','중형','대형']
    ))
)

In [57]:
# 열별 이용규모_그룹 별 이용비율 분포 확인
pd.crosstab([station_df['월별코드'],station_df['이용규모_그룹']], station_df['이용비율_그룹'], normalize = 'index')

이용비율_그룹              0      high       low    middle
월별코드 이용규모_그룹                                        
2501 소형       0.699422  0.104046  0.086705  0.109827
     중형       0.401163  0.122093  0.261628  0.215116
     대형       0.219653  0.028902  0.531792  0.219653
2502 소형       0.699422  0.104046  0.086705  0.109827
     중형       0.406977  0.081395  0.279070  0.232558
     대형       0.213873  0.028902  0.566474  0.190751
2503 소형       0.699422  0.167630  0.040462  0.092486
     중형       0.406977  0.220930  0.133721  0.238372
     대형       0.219653  0.075145  0.404624  0.300578
2504 소형       0.693642  0.208092  0.023121  0.075145
     중형       0.421965  0.277457  0.086705  0.213873
     대형       0.208092  0.179191  0.358382  0.254335
2505 소형       0.693642  0.213873  0.017341  0.075145
     중형       0.421965  0.329480  0.075145  0.173410
     대형       0.213873  0.202312  0.317919  0.265896
2506 소형       0.699422  0.208092  0.011561  0.080925
     중형       0.410405  0.329480  0.069364  0.190751
     대형       0.219653  0.248555  0.271676  0.260116
2507 소형       0.712644  0.172414  0.022989  0.091954
     중형       0.401163  0.296512  0.098837  0.203488
     대형       0.218391  0.172414  0.327586  0.281609
2508 소형       0.705202  0.202312  0.017341  0.075145
     중형       0.410405  0.306358  0.086705  0.196532
     대형       0.219653  0.219653  0.294798  0.265896
2509 소형       0.710983  0.190751  0.005780  0.092486
     중형       0.406977  0.319767  0.081395  0.191860
     대형       0.208092  0.231214  0.300578  0.260116
2510 소형       0.693642  0.196532  0.023121  0.086705
     중형       0.416185  0.289017  0.086705  0.208092
     대형       0.213873  0.196532  0.329480  0.260116
2511 소형       0.705202  0.184971  0.023121  0.086705
     중형       0.416185  0.242775  0.104046  0.236994
     대형       0.202312  0.115607  0.381503  0.300578
2512 소형       0.699422  0.104046  0.075145  0.121387
     중형       0.412791  0.087209  0.279070  0.220930
     대형       0.208092  0.040462  0.537572  0.213873

In [58]:
# 월 빼고 이용규모 별 따릉이 이용비율 분포
overall = pd.crosstab(
    station_df['이용규모_그룹'],
    station_df['이용비율_그룹'],
    normalize='index'
)
# 비율[%]
(overall * 100).round(1)

이용비율_그룹,0,high,low,middle
이용규모_그룹,,,,
소형,70.1,17.1,3.6,9.1
중형,41.1,24.2,13.7,21.0
대형,21.4,14.5,38.5,25.6


반경 300m 이내에 따릉이 대여소가 없는 비율이 제일 높은 역은 소형그룹으로 70.1%이다. 이는 대,중 그룹보다 높은 비율이다. 지하철이용규모 대비 따릉이 이용비율이 높은 그룹 비중은 중형그룹이 24.2%로 높게 나타났다. 지하철 이용규모 대비 따릉이 이용비율이 낮은 그룹 비중은 대형그룹이 38.5%로 높게 나왔다.

## 따릉이 이용 비율 별 300m 내 따릉이 이용건수 비교

In [59]:
# 피벗테이블 생성 (월별코드 x 이용규모그룹 x 이용비율그룹별 300m 따릉이 이용건수 합)
pivot_result_month = station_df.pivot_table(
    index=['월별코드', '이용규모_그룹'],
    columns='이용비율_그룹',
    values='300m_따릉이이용건수',
    aggfunc='sum'
)
pivot_result_month

/var/folders/5y/smp8bthj4zg0txjv4pn26xb80000gn/T/ipykernel_28653/3030708539.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_result_month = station_df.pivot_table(


이용비율_그룹       0    high     low  middle
월별코드 이용규모_그룹                           
2501 소형       0   37038    4798   16024
     중형       0   95577   33192   80725
     대형       0   43825  176779  160662
2502 소형       0   35176    4205   16220
     중형       0   67553   35008   89558
     대형       0   40051  206914  126394
2503 소형       0   90262    2333   17468
     중형       0  223716   19801   98812
     대형       0  141541  192607  327989
2504 소형       0  131216    1573   12872
     중형       0  317670   13418   91176
     대형       0  330245  197949  320639
2505 소형       0  144497     837   13038
     중형       0  379780   10751   72262
     대형       0  385226  168335  344741
2506 소형       0  139717     858   13060
     중형       0  398749   10645   78666
     대형       0  467169  139497  326501
2507 소형       0  104344    1006   14874
     중형       0  302150   15690   82796
     대형       0  317350  177017  347370
2508 소형       0  119106     988    9450
     중형       0  319473   11915   77170
     대형       0  376901  141541  315015
2509 소형       0  135623     489   15349
     중형       0  360747   12098   81195
     대형       0  464427  152551  332973
2510 소형       0  117837    1047   13486
     중형       0  289916   14950   78997
     대형       0  348941  160007  310519
2511 소형       0  100716    1445   13423
     중형       0  241261   16304   99286
     대형       0  227603  192088  348798
2512 소형       0   42498    4111   21351
     중형       0   75262   43293   93721
     대형       0   78514  222543  175990

In [60]:
# 월 구분 없이: 이용규모그룹 x 이용비율그룹별 300m 따릉이 이용건수 합(비율)
pivot_result = station_df.pivot_table(
    index='이용규모_그룹',
    columns='이용비율_그룹',
    values='300m_따릉이이용건수',
    aggfunc='sum'
)

pivot_ratio = pivot_result.div(pivot_result.sum(axis=1), axis=0) * 100
pivot_ratio = pivot_ratio.round(1)
pivot_ratio

/var/folders/5y/smp8bthj4zg0txjv4pn26xb80000gn/T/ipykernel_28653/1833785727.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_result = station_df.pivot_table(


이용비율_그룹,0,high,low,middle
이용규모_그룹,,,,
소형,0.0,85.7,1.7,12.6
중형,0.0,70.9,5.5,23.6
대형,0.0,36.7,24.2,39.1


피벗 결과를 통해 **역 규모(대형/중형/소형) 안에서** 이용비율(S~D 등급)이 높은 그룹일수록 **300m 반경 내 따릉이 이용건수(합)**가 어떻게 달라지는지 비교한다.

In [61]:
station_df.to_excel('이용규모별이용비율.xlsx')

# 가설 검정(카이제곱 검정) 01
귀무가설 : 지하철 이용규모와 따릉이 이용비율은 서로 독립이다.  
대립가설 : 지하철 이용규모와 따릉이 이용비율은 서로 독립이 아니다.

In [62]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    station_df['이용규모_그룹'],
    station_df['이용비율_그룹']
)

chi2, p, dof, expected = chi2_contingency(table)

chi2, p


(np.float64(1502.5903860916956), np.float64(0.0))

# 가설 검정(피어슨 상관계수 검정) 02
귀무가설 : 하차인원 대비 따릉이 이용비율과 300m 내 따릉이 이용건수는 서로 독립이다.

대립가설 : 하차인원 대비 따릉이 이용비율과 300m 내 따릉이 이용건수는 서로 독립이 아니다.

In [63]:
import scipy.stats as stats

station_df['하차인원대비따릉이이용비율[%]'] = station_df['하차인원대비따릉이이용비율[%]'].fillna(0)

# 이용 규모 그룹별: (하차인원 대비 따릉이 이용비율)과 (300m 내 따릉이 이용건수) 상관
for grp in ['대형','중형','소형']:
    df_g = station_df[station_df['이용규모_그룹'] == grp]
    # 상관 계산은 결측 제거 후 수행
    x = df_g['하차인원대비따릉이이용비율[%]']
    y = df_g['300m_따릉이이용건수']
    mask = x.notna() & y.notna()
    if mask.sum() >= 2:
        corr, p = stats.pearsonr(x[mask], y[mask])
        r2 = corr**2
        print(grp, corr, r2, p)
    else:
        print(grp, '데이터 부족')


대형 0.8286513311733283 0.6866630286553291 0.0
중형 0.9539557269290023 0.9100315289406412 0.0
소형 0.9115578042561523 0.8309376305002977 0.0


# 결론
01: p-value가 유의수준(0.05)보다 작으므로 귀무가설을 기각한다. 즉, 지하철 이용 규모와 따릉이 이용 비율 간 통계적으로 유의한 관계가 있다. 중형 규모 역에서는 따릉이 이용 비율 비중이 높고 대형 규모 역에서는 따릉이 이용 비율 비중이 낮다. 대형 규모 역은 하차 승객 대비 따릉이 이용률이 낮으므로 편의성을 점검해야한다.

02: (역 규모 그룹별) 하차인원 대비 따릉이 이용비율과 300m 내 따릉이 이용건수의 상관을 확인했다. 상관계수(r)와 p-value를 함께 보고, 규모별로 이용 패턴이 유사한지 비교한다.

# 📂 지표정규화_상대비교.ipynb

### 지표 정규화 & 상대 비교

#### 라이브러리 / 경로
역별 따릉이 이용 수준을 공정하게 비교하기 위해 하차 인원으로 보정한 지표를 만들고, 같은 조건(월/규모그룹) 안에서 상대 비교를 수행한다.

In [64]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
base_path = os.getcwd()

PATH_USAGE_LOG = os.path.join(base_path, '따릉이이용로그월별시트.xlsx')
PATH_SUBWAY_INFO = os.path.join(
    base_path, 'final_station_scale_analysis_거치대수추가.csv') # 역 단위 고정 정보
PATH_SUBWAY_MONTHLY = os.path.join(base_path, 'subway_2025.xlsx') # 월별 하차인원 정보
PATH_RELATIVE_RESULT = os.path.join(base_path, 'station_monthly_relative_result.csv')

PATH_USAGE_LOG, PATH_SUBWAY_INFO, PATH_SUBWAY_MONTHLY, PATH_RELATIVE_RESULT

('/Users/hyw/Desktop/코드잇/초급 프로젝트/따릉이이용로그월별시트.xlsx',
 '/Users/hyw/Desktop/코드잇/초급 프로젝트/final_station_scale_analysis_거치대수추가.csv',
 '/Users/hyw/Desktop/코드잇/초급 프로젝트/subway_2025.xlsx',
 '/Users/hyw/Desktop/코드잇/초급 프로젝트/station_monthly_relative_result.csv')

#### 데이터 로드 및 컬럼 확인


따릉이 이용 데이터는 '대여소' 기준으로 되어 있고, '지하철역' 단위로 비교해야 한다.

이후 단계에서 대여소번호 -> 지하철역 매핑, 역x월 단위 집계를 통해 비교 단위를 맞춘다.

In [65]:
sheets = pd.read_excel(PATH_USAGE_LOG, sheet_name=None)
usage_raw = pd.concat(sheets.values(), ignore_index=True)

# 역 단위 고정 정보
station_info_df = pd.read_csv(PATH_SUBWAY_INFO)

usage_raw.shape, station_info_df.shape

((1230412, 11), (858, 8))

#### 대여소번호 -> 지하철역 매핑


- 따릉이 이용 로그는 대여소번호 기준이고, 역 규모 정보는 지하철역 기준이다.

- 두 데이터를 같은 단위로 맞추기 위해 대여소번호를 지하철역으로 연결한 뒤 분석을 진행한다.

- 이 단계에서는 merge가 안정적으로 되도록 타입을 맞추고, 매핑 누락이 발생하는지 확인한다.

In [66]:
# 데이터 오염 가능성이 있으므로 원본 다시 로드
station_info_df = pd.read_csv(PATH_SUBWAY_INFO)

In [67]:
# 타입 통일
usage_raw['대여소번호'] = pd.to_numeric(usage_raw['대여소번호'], errors='coerce')
station_info_df['대여소번호'] = pd.to_numeric(station_info_df['대여소번호'], errors='coerce')

# 매핑 테이블 생성
map_df = (
    station_info_df[['대여소번호', '지하철역']].dropna().drop_duplicates()
)

map_df.shape, map_df.head()

((858, 2),
     대여소번호      지하철역
 0  1539.0  4.19민주묘지
 1  1564.0  4.19민주묘지
 2  1568.0  4.19민주묘지
 3  1201.0      가락시장
 4  1203.0      가락시장)

In [68]:
# 대여소번호가 여러 지하철역으로 연결되지 확인
dup_station_cnt = (map_df.groupby('대여소번호')['지하철역'].nunique().sort_values(ascending=False))

dup_station_cnt.head(10)

대여소번호
6190.0    2
2060.0    2
318.0     2
3401.0    2
4870.0    2
4865.0    2
2230.0    2
3416.0    2
361.0     2
1268.0    2
Name: 지하철역, dtype: int64

In [69]:
usage_m = usage_raw.copy()
usage_m['month'] = usage_m['대여일자'].astype(str).str.slice(4, 6).astype(int)

usage_m = usage_m.merge(map_df, on='대여소번호', how='left')

usage_m['지하철역'].isna().mean()

np.float64(0.6821272075957854)

In [70]:
usage_m['month'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

#### 역x월 이용량 집계

- 월별로 역 단위 이용량(이용건수 합)을 만든다.
- 이 결과는 이후 지표 생성에서 분자(bike_usage)가 된다.
- 집계 결과가 지하철역xmonth 조합 기준으로 중복 없이 정리됐는지도 같이 확인한다.

In [71]:
# 월x역 이용건수(합) 집계
station_monthly_usage = (
    usage_m.dropna(subset=['지하철역']).groupby(['지하철역', 'month'], as_index=False)['이용건수']
    .sum().rename(columns={'이용건수': 'bike_usage'})
)

station_monthly_usage.duplicated(['지하철역', 'month']).sum(), station_monthly_usage.head()

(np.int64(0),
        지하철역  month  bike_usage
 0  4.19민주묘지      1         767
 1  4.19민주묘지      2         762
 2  4.19민주묘지      3        1550
 3  4.19민주묘지      4        2083
 4  4.19민주묘지      5        2283)

#### 역 규모 정보 결합

이용량(bike_usage)만 비교하면 역 규모가 큰 곳이 유리해진다. 그래서 역별 하차 인원(규모)과 규모그룹 정보를 결합한다.


scale_df는 대여소 단위로 구성되어 있어 동일한 지하철역 정보가 여러 행에 반복된다. 이 중 지하철하차인원과 규모그룹은 같은 지하철역 내에서 값이 변하지 않는 것이 확인되어 역 기준으로 중복을 정리한 뒤 결합에 사용했다.


In [72]:
xls = pd.ExcelFile(PATH_SUBWAY_MONTHLY)

dfs = []
for sh in xls.sheet_names:            # '2501'~'2512'
    tmp = pd.read_excel(PATH_SUBWAY_MONTHLY, sheet_name=sh)
    tmp["month"] = int(sh[-2:])       # 1~12
    dfs.append(tmp)

subway_all = pd.concat(dfs, ignore_index=True)

# 월별 하차인원 (역 × month)
monthly_alight = (
    subway_all.groupby(["역명", "month"], as_index=False)["하차총승객수"].sum()
    .rename(columns={"역명": "지하철역", "하차총승객수": "지하철하차인원"})
)

# 규모그룹(역 단위 고정) 붙이기
scale_group = station_info_df[["지하철역", "규모그룹"]].drop_duplicates()

station_info = monthly_alight.merge(scale_group, on="지하철역", how="left")

station_info.shape, station_info.head()

((6345, 4),
        지하철역  month  지하철하차인원    규모그룹
 0  4.19민주묘지      1    74425  소형(여유)
 1  4.19민주묘지      2    75772  소형(여유)
 2  4.19민주묘지      3   120713  소형(여유)
 3  4.19민주묘지      4   124599  소형(여유)
 4  4.19민주묘지      5   118469  소형(여유))

In [73]:
# 결측 비율 확인
station_monthly = station_monthly_usage.merge(
    station_info, on=["지하철역", "month"], how="left"
)
station_monthly.shape, station_monthly[['지하철하차인원', '규모그룹']].isna().mean()

((3124, 5),
 지하철하차인원    0.025608
 규모그룹       0.025608
 dtype: float64)

In [74]:
print("after merge:", station_monthly['month'].unique())

after merge: [ 1  2  3  4  5  6  7  8  9 10 11 12]


#### 지표생성: 하차 인원 대비 따릉이 이용 비율

역 규모 차이를 줄이기 위해, 하차 인원 대비 따릉이 이용 비율(usage_ratio)을 지표로 사용한다.

- 분자: 역x월 따릉이 이용량(bike_usage)
- 분모: 역 하차 인원(지하철하차인원)

이 지표를 통해 단순 이용량이 아닌, 규모를 고려한 이용 수준을 비교

In [75]:
# 하차 인원 대비 따릉이 이용 비율(정규화 지표)
station_monthly['usage_ratio'] = station_monthly['bike_usage'] / station_monthly['지하철하차인원']
station_monthly[['지하철역', 'month', '규모그룹', 'bike_usage', '지하철하차인원', 'usage_ratio']].head()

,지하철역,month,규모그룹,bike_usage,지하철하차인원,usage_ratio
0,4.19민주묘지,1,소형(여유),767,74425.0,0.010306
1,4.19민주묘지,2,소형(여유),762,75772.0,0.010056
2,4.19민주묘지,3,소형(여유),1550,120713.0,0.012840
3,4.19민주묘지,4,소형(여유),2083,124599.0,0.016718
4,4.19민주묘지,5,소형(여유),2283,118469.0,0.019271


#### 월별 분포 확인


월마다 전체 이요 수준(usage_ratio)의 분포가 달라질 수 있다.

월을 섞어서 비교하면, 월 효과가 함께 들어가 직접 비교가 어려워진다.

상대 비교는 같은 월 안에서 수행한다.

In [76]:
station_monthly.groupby('month')['usage_ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
month,,,,,,,,
1,255.0,0.007916,0.016947,0.000311,0.002555,0.004723,0.008253,0.242489
2,255.0,0.007314,0.016058,0.000275,0.002367,0.004432,0.007561,0.231577
3,254.0,0.011491,0.023060,0.000507,0.003702,0.007206,0.012740,0.317645
4,254.0,0.014384,0.028105,0.000601,0.004765,0.009181,0.016180,0.382015
5,253.0,0.015610,0.030245,0.000688,0.004964,0.009805,0.017157,0.406061
6,253.0,0.017133,0.032956,0.000788,0.005533,0.010790,0.019333,0.447595
7,253.0,0.014271,0.028990,0.000690,0.004636,0.009037,0.015693,0.409007
8,252.0,0.015945,0.031630,0.000773,0.005124,0.010029,0.017806,0.441457
9,253.0,0.015801,0.032462,0.000259,0.005246,0.010073,0.017098,0.458450


-  usage_ratio 값 범위가 최소-최대 차이가 매우 큼
- 상위 일부 역이 분포를 강하게 끌어올리는 구조
- 절대값 비교만으로 '좋다/나쁘다' 판단하기 어려움

#### 월 x 규모그룹 상대순위 산정

- 정규화(usage_ratio)를 했더라도, 대형역과 소형역은 기본 분포 자체가 다를 수 있다.
- 같은 월(month), 같은 규모그룹 안에서만 순위를 매겨 상대 비교를 수행한다.

In [77]:
# 같은 월, 같은 규모그룹 안에서 usage_ratio 기준 상대순위
station_monthly['relative_rank'] = (
    station_monthly.groupby(['month', '규모그룹'])['usage_ratio'].rank(ascending=False, method='min')
)

# 같은 그룹 내 역 개수(등급 컷 계산용)
station_monthly['group_size'] = (
    station_monthly.groupby(['month', '규모그룹'])['지하철역'].transform('count')
)

station_monthly.groupby(['month', '규모그룹', 'usage_ratio', 'relative_rank', 'group_size']).size().head()

month  규모그룹    usage_ratio  relative_rank  group_size
1      대형(혼잡)  0.000311     86.0           86.0          1
               0.000337     85.0           86.0          1
               0.000475     84.0           86.0          1
               0.000497     83.0           86.0          1
               0.000640     82.0           86.0          1
dtype: int64

#### 등급화(S~D)

- s: 상위 20%
- A: 20~40%
- B: 40~60%
- C: 60~80%
- D: 하위 20%

In [78]:
def assign_grade(rank, group_size):
    p = rank / group_size
    if p <=0.2:
        return 'S'
    elif p <= 0.4:
        return 'A'
    elif p <= 0.6:
        return 'B'
    elif p <= 0.8:
        return 'C'
    else:
        return 'D'
    
station_monthly['grade'] = station_monthly.apply(
    lambda x: assign_grade(x['relative_rank'], x['group_size']), axis=1
)

station_monthly[['지하철역', 'month', '규모그룹', 'usage_ratio', 'relative_rank', 'grade']].head()

,지하철역,month,규모그룹,usage_ratio,relative_rank,grade
0,4.19민주묘지,1,소형(여유),0.010306,30.0,A
1,4.19민주묘지,2,소형(여유),0.010056,29.0,A
2,4.19민주묘지,3,소형(여유),0.012840,41.0,B
3,4.19민주묘지,4,소형(여유),0.016718,38.0,B
4,4.19민주묘지,5,소형(여유),0.019271,34.0,B


#### 결과 저장 및 요약

최종 결과를 저장하고, 특정 월 기준으로 상・하위 역을 확인해 결과가 자연스럽게 보이는지 검토한다.

In [79]:
station_monthly.to_csv(PATH_RELATIVE_RESULT, index=False)
PATH_RELATIVE_RESULT

'/Users/hyw/Desktop/코드잇/초급 프로젝트/station_monthly_relative_result.csv'

In [80]:
view = station_monthly[station_monthly['month'] == 1].copy()

top10 = view.sort_values('usage_ratio', ascending=False).head(10)
bottom10 = view.sort_values('usage_ratio', ascending=True).head(10)

top10[['지하철역', '규모그룹', 'usage_ratio', 'grade']], bottom10[['지하철역', '규모그룹', 'usage_ratio', 'grade']]

(            지하철역    규모그룹  usage_ratio grade
 300         공항시장  소형(여유)     0.242489     S
 2968        한성백제  소형(여유)     0.070508     S
 946           마곡  소형(여유)     0.055316     S
 2368   용두(동대문구청)  소형(여유)     0.051487     S
 2788          창신  소형(여유)     0.040549     S
 1504         서강대  소형(여유)     0.039043     S
 1900         신방화  소형(여유)     0.038626     S
 1090  몽촌토성(평화의문)  소형(여유)     0.027248     S
 60            가좌  소형(여유)     0.022575     S
 1876         신목동  소형(여유)     0.022040     S,
         지하철역    규모그룹  usage_ratio grade
 2068  압구정로데오  대형(혼잡)     0.000311     D
 1336      사당  대형(혼잡)     0.000337     D
 1624      선릉  대형(혼잡)     0.000475     D
 84      강남구청  대형(혼잡)     0.000497     D
 2200      역삼  대형(혼잡)     0.000640     D
 72        강남  대형(혼잡)     0.000645     D
 1564     서울역  대형(혼잡)     0.000672     D
 2776      창동  대형(혼잡)     0.000699     D
 2944     한강진  중형(보통)     0.000707     D
 2920      학동  대형(혼잡)     0.000721     D)